# Hotel Booking Cancellation Prediction

## 1 - Dataset Selection & Understanding

### Objective

This notebook documents the selection and initial understanding of the validated Hotel Booking dataset used for the IT3051 FDM Mini Project.

The main objectives are to:

1. Understand the real-world hotel booking cancellation problem.
2. Document the approved dataset source and dataset description.
3. Identify the target variable, features, and feature types.
4. Create a complete data dictionary for the relevant attributes.
5. Examine basic numerical statistics and categorical distributions.
6. Investigate missing values, duplicates, invalid/unusual values, and potential leakage variables.
7. Confirm the suitability of the dataset for hotel booking cancellation prediction.

### Selected Dataset

The selected dataset is the lecturer-validated `hotel_booking.csv` dataset.

Each row represents a hotel booking record. The columns contain information about the hotel type, booking timing, arrival date, length of stay, guest counts, meal plan, country, market segment, distribution channel, previous booking behavior, room type, deposit type, customer type, average daily rate, reservation status, and selected guest identifier fields.

The target variable is `is_canceled`, which indicates whether a booking was canceled.

## 1. Problem Definition

Hotel booking cancellations create operational and financial challenges for hotels. When a confirmed booking is canceled, the hotel may lose expected revenue and may also face difficulties with room allocation, staffing, inventory planning, and demand forecasting. Understanding cancellation patterns helps hotels make better decisions about pricing, overbooking, and customer management.

## 2. Prediction Objective

The prediction objective is to use historical booking information to predict whether a hotel booking will be canceled. This is a supervised binary classification problem because the output has two classes: canceled and not canceled.

## 3. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 120)

## 4. Load the Approved Raw Dataset

The approved raw dataset is stored in the project folder under `data/raw/hotel_booking.csv`. The raw dataset is loaded without preprocessing or transformation.

In [ ]:
DATA_PATH = Path('../data/raw/hotel_booking.csv')
df = pd.read_csv(DATA_PATH)
print('Dataset loaded successfully')

## 5. Preview the Dataset

The first and last records are displayed to understand the general appearance of the dataset.

In [ ]:
df.head()

In [ ]:
df.tail()

## 6. Dataset Shape and General Structure

This section identifies the number of records, number of columns, and the general structure of the dataset.

In [ ]:
dataset_shape = pd.DataFrame({
    'Property': ['Number of Records', 'Number of Columns'],
    'Value': [df.shape[0], df.shape[1]]
})
dataset_shape

## 7. Column Information

The following table shows all attributes and their pandas data types. This helps identify numerical, categorical, binary, date/time, and identifier fields.

In [ ]:
pd.DataFrame({'column': df.columns, 'pandas_dtype': df.dtypes.astype(str).values})

## 8. Target Variable Identification

The target variable is `is_canceled`.

- `0` means the booking was not canceled.
- `1` means the booking was canceled.

This target makes the project a binary classification problem.

In [ ]:
target_distribution = (
    df['is_canceled']
    .value_counts()
    .sort_index()
    .rename_axis('is_canceled')
    .reset_index(name='count')
)
target_distribution['meaning'] = target_distribution['is_canceled'].map({0: 'Not canceled', 1: 'Canceled'})
target_distribution['percentage'] = (target_distribution['count'] / len(df) * 100).round(3)
target_distribution

## 9. Feature and Attribute Type Summary

The dataset contains numerical, categorical, binary, date/time, and identifier/PII attributes. The target variable is included in the binary category.

In [ ]:
data_dictionary = pd.read_csv('data_dictionary.csv')
feature_type_summary = (
    data_dictionary['attribute_type']
    .value_counts()
    .rename_axis('Attribute Type')
    .reset_index(name='Count')
    .sort_values('Attribute Type')
)
feature_type_summary

## 10. Data Dictionary

The separate `data_dictionary.csv` file is kept as a reusable artifact because it provides a clean reference for all 36 attributes. It includes the attribute name, description, attribute type, pandas data type, missing-value information, number of unique values, example values, and important notes.

In [ ]:
data_dictionary = pd.read_csv('data_dictionary.csv')
data_dictionary

## 11. Numerical Descriptive Statistics

The following table provides initial descriptive statistics for numerical attributes. These statistics help understand central tendency, spread, minimum values, maximum values, and possible unusual values.

In [ ]:
numeric_columns = [col for col in df.select_dtypes(include='number').columns if col != 'is_canceled']
df[numeric_columns].describe().T.round(3)

## 12. Categorical Value Investigation

This section examines unique values and basic category distributions for selected categorical attributes. The purpose is to understand dominant categories and identify unusual labels such as `Undefined`.

In [ ]:
categorical_columns = [
    'hotel', 'meal', 'country', 'market_segment', 'distribution_channel',
    'reserved_room_type', 'assigned_room_type', 'deposit_type',
    'customer_type', 'reservation_status'
]

categorical_summary = []
for column in categorical_columns:
    categorical_summary.append({
        'attribute': column,
        'unique_values': df[column].nunique(dropna=True),
        'top_value': df[column].mode(dropna=True).iloc[0],
        'top_value_count': int(df[column].value_counts(dropna=True).iloc[0]),
        'top_value_percentage': round(df[column].value_counts(dropna=True).iloc[0] / len(df) * 100, 3)
    })

pd.DataFrame(categorical_summary)

In [ ]:
for column in categorical_columns:
    print(f'\n{column}')
    display(df[column].value_counts(dropna=False).head(10).to_frame('count'))

## 13. Missing Value Investigation

This section identifies attributes with missing values. At the dataset understanding stage, the objective is to observe and document missingness, not to perform imputation or cleaning.

In [ ]:
missing_summary = df.isna().sum().reset_index()
missing_summary.columns = ['attribute', 'missing_count']
missing_summary['missing_percentage'] = (missing_summary['missing_count'] / len(df) * 100).round(3)
missing_summary = missing_summary[missing_summary['missing_count'] > 0]
missing_summary.sort_values('missing_count', ascending=False)

## 14. Duplicate Investigation

This section checks whether the dataset contains fully duplicated records.

In [ ]:
pd.DataFrame({'metric': ['Fully duplicated records'], 'value': [int(df.duplicated().sum())]})

## 15. Invalid / Unusual Values

This section performs an initial check for values that may require deeper investigation later. No cleaning is performed here because this stage is limited to dataset understanding.

In [ ]:
unusual_values = pd.DataFrame([
    {'check': 'Negative ADR values', 'count': int((df['adr'] < 0).sum())},
    {'check': 'Zero total guests', 'count': int(((df['adults'] + df['children'].fillna(0) + df['babies']) == 0).sum())},
    {'check': 'Zero total nights', 'count': int(((df['stays_in_weekend_nights'] + df['stays_in_week_nights']) == 0).sum())},
    {'check': 'Undefined meal values', 'count': int((df['meal'] == 'Undefined').sum())},
    {'check': 'Undefined market segment values', 'count': int((df['market_segment'] == 'Undefined').sum())},
    {'check': 'Undefined distribution channel values', 'count': int((df['distribution_channel'] == 'Undefined').sum())},
])
unusual_values

## 16. Initial Class Distribution

The target-variable distribution shows whether the cancellation classes are balanced or imbalanced. This is useful for later modeling decisions, but no modeling is performed in this stage.

In [ ]:
target_distribution

## 17. Potential Data Leakage

Potential leakage variables are attributes that may reveal the booking outcome directly or may only be known after the cancellation decision has already happened. These variables should be carefully reviewed before future modeling.

In [ ]:
potential_leakage = pd.DataFrame([
    {'attribute': 'reservation_status', 'reason': 'Directly represents the final booking outcome.'},
    {'attribute': 'reservation_status_date', 'reason': 'May be known only after cancellation/check-out status is confirmed.'},
    {'attribute': 'assigned_room_type', 'reason': 'May be assigned after operational processing and should be reviewed before modeling.'},
])
potential_leakage

## 18. Dataset Characteristics Summary

The following table summarizes the main dataset characteristics required for the Dataset Selection & Understanding stage.

In [ ]:
dataset_summary = {
    'Dataset Name': 'Hotel Booking Dataset',
    'Dataset Status': 'Lecturer-validated raw dataset',
    'Number of Records': df.shape[0],
    'Number of Columns': df.shape[1],
    'Target Variable': 'is_canceled',
    'Prediction Type': 'Binary classification',
    'Canceled Bookings': int((df['is_canceled'] == 1).sum()),
    'Non-Canceled Bookings': int((df['is_canceled'] == 0).sum()),
    'Columns With Missing Values': int((df.isna().sum() > 0).sum()),
    'Duplicate Rows': int(df.duplicated().sum())
}

summary_df = pd.DataFrame(dataset_summary.items(), columns=['Property', 'Value'])
summary_df

## 19. Final Overview

In [ ]:
print('=' * 70)
print('HOTEL BOOKING DATASET - FINAL OVERVIEW')
print('=' * 70)

print('Dataset: Hotel Booking Dataset')
print(f'Records: {df.shape[0]}')
print(f'Columns: {df.shape[1]}')
print('Target: is_canceled')
print('Problem Type: Binary Classification')

print('Target Distribution:')
print(target_distribution[['meaning', 'count', 'percentage']])

print('\nMissing Values by Column:')
print(missing_summary.sort_values('missing_count', ascending=False))

print('\nDuplicate Rows:')
print(df.duplicated().sum())

print('\nPotential Leakage Variables:')
print(potential_leakage)
print('=' * 70)

## 20. Dataset Suitability

The dataset is suitable for the Hotel Booking Cancellation Prediction problem because it contains a clear binary target variable, a large number of booking records, and multiple relevant predictor attributes related to booking timing, stay details, guest composition, customer history, market channel, deposit type, and reservation behavior. It also contains realistic data-quality conditions such as missing values and unusual records, which makes it appropriate for Evaluation 1 tasks such as data understanding, initial investigation, and preparation for later EDA and preprocessing.

## 21. Conclusion

The lecturer-validated Hotel Booking dataset was selected as the approved raw dataset for the Hotel Booking Cancellation Prediction project.

The dataset contains 119,390 booking records and 36 columns. Each record represents a hotel booking, and the target variable is `is_canceled`, where `0` represents a non-canceled booking and `1` represents a canceled booking. Therefore, the project is suitable for a supervised binary classification problem.

The initial dataset understanding stage identified the main dataset structure, attribute meanings, feature types, target distribution, missing-value conditions, duplicate status, unusual values, and potential leakage variables. The raw dataset has been preserved without preprocessing or transformation. The data dictionary and dataset characteristics summary have been prepared for reference in the next stages of the project.